# Telemanom veri analizi

Esra — dataset & preprocessing

Bu notebook’ta önce veriye göz atıyorum, sonra kendi yazdığım `preprocessing.py` ve `data_loader.py` ile pipeline’ı deniyorum. Veri: [appleparan/telemanom](https://huggingface.co/datasets/appleparan/telemanom).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from data_loader import (
    build_telemanom_pipeline,
    create_dataloaders,
    list_hf_channels,
    load_channel_for_analysis,
    load_hf_labels,
)
from preprocessing import (
    PreprocessConfig,
    anomaly_mask,
    class_distribution,
    sequence_length_stats,
)

sns.set_theme(style="whitegrid")

## Veriye ilk bakış

Kaç kanal var, uzay aracı dağılımı ve anomali tipleri (point / contextual) nasıl?

In [ ]:
labels = load_hf_labels()
channels = list_hf_channels()
print(f"Toplam kanal: {len(channels)}")
labels.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels["spacecraft"].value_counts().plot(kind="bar", ax=axes[0], title="SMAP vs MSL")
labels["class"].value_counts().plot(kind="bar", ax=axes[1], title="Anomali tipi")
plt.tight_layout()
plt.show()

## Sekans uzunlukları

Kanalların uzunluğu çok farklı; aşağıda ilk 10 kanaldan örnek aldım. Bu yüzden sabit uzunluk için sliding window kullandım.

In [ ]:
lengths = []
for cid in channels[:10]:
    train, test, _ = load_channel_for_analysis(cid)
    lengths.extend([len(train), len(test)])

display(sequence_length_stats(lengths))
pd.DataFrame({"timesteps": lengths}).boxplot()
plt.title("İlk 10 kanal — zaman adımı sayısı")
plt.show()

## Normal vs anomali — örnek kanal

Kırmızı bölgeler etiketli anomali aralıkları. Train tarafı zaten normal kabul ediliyor.

In [ ]:
cid = channels[0]
train, test, row = load_channel_for_analysis(cid)
target = test[:, 0]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(target, linewidth=0.8, label="telemetri")
if row is not None:
    mask = anomaly_mask(len(target), row["anomaly_sequences"])
    ax.fill_between(
        np.arange(len(target)), target.min(), target.max(),
        where=mask, alpha=0.3, color="red", label="anomali",
    )
ax.set_title(f"Test akışı — kanal {cid}")
ax.legend()
plt.show()

## Ön işleme pipeline’ı

StandardScaler, 50’lik pencere, stride 10, stratified train/val/test. Notebook hızlı kalsın diye 15 kanal; tam veri için `max_channels=None`.

In [ ]:
config = PreprocessConfig(sequence_length=50, stride=10, random_state=42)
result = build_telemanom_pipeline(config=config, max_channels=15)

info = result.model_interface.as_dict()
print("Model ekibine not:")
for k, v in info.items():
    print(f"  {k}: {v}")

## Dengesizlik

Pencere etiketlerinde normal çok daha fazla
 stratified split şart.

In [ ]:
all_y = np.concatenate([result.y_train, result.y_val, result.y_test])
dist = class_distribution(all_y)
display(dist)

ratio = dist.attrs.get("imbalance_ratio")
if ratio:
    print(f"Çoğunluk/azınlık oranı yaklaşık {ratio:.1f}:1")

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(data=dist, x="class_name", y="count", ax=ax)
ax.set_title("Pencere etiketleri")
plt.show()

## DataLoader kontrolü

Batch’ler tensor olarak geliyor mu, son bir bakış.

In [ ]:
loaders = create_dataloaders(
    result.x_train, result.y_train,
    result.x_val, result.y_val,
    result.x_test, result.y_test,
    batch_size=32,
)
bx, by = next(iter(loaders["train"]))
print("Batch boyutu:", tuple(bx.shape))
print("İlk etiketler:", by[:8].tolist())